# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

1. The database has 91.26% legit(0) vs 8.74% fraud(1). That's roughly a 10.4:1 ratio - yes, this is a clearly imbalanced dataset. With this imbalance, accuracy alone is misleading.

In [ ]:
# Step 2
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
def evaluate(model, X_te, y_te, label):
    """Evaluate a fitted classifier with imbalance-aware metrics."""
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    print(classification_report(y_te, y_pred, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_te, y_pred))
    print(f"Precision (fraud=1): {precision_score(y_te, y_pred):.4f}")
    print(f"Recall (fraud=1): {recall_score(y_te, y_pred):.4f}")
    print(f"F1 (fraud=1): {f1_score(y_te, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}")
    print(f"PR-AUC (avg prec): {average_precision_score(y_te, y_proba):.4f}")

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_s, y_train)
evaluate(log_reg, X_test_s, y_test, "BASELINE (imbalanced)")

In [ ]:
# Step 4
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_s, y_train)
print("\nOversampled train distribution:\n", y_train_ros.value_counts())
log_reg_ros = LogisticRegression(max_iter=1000, random_state=42)
log_reg_ros.fit(X_train_ros, y_train_ros)
evaluate(log_reg_ros, X_test_s, y_test, "OVERSAMPLED (RandomOverSampler)")

In [ ]:
#Step 5
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_s, y_train)
print("\nUndersampled train distribution:\n", y_train_rus.value_counts())
log_reg_rus = LogisticRegression(Max_iter=1000, random_state=42)
log_reg_rus.fit(X_train_rus, y_train_rus)
evaluate(log_reg_rus, X_test_s, y_test, "UNDERSAMPLED (RandomUnderSampler)")

In [ ]:
# Step 6
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)
print("\nSMOTE train distribution:\n", y_train_sm.value_counts())
log_reg_sm = LogisticRegression(max_iter=1000, random_state=42)
log_reg_sm.fit(X_train_sm, y_train_sm)
evaluate(log_reg_sm, X_test_sm, y_test, "SMOTE")